In [13]:
# imports
import json
import os
import numpy as np
from tqdm import tqdm
from google import genai
from google.genai import types
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(r"C:\Users\ADMIN\Documents\GitHub\eschool-chatbot\embeddinggemma\embeddinggemma-300m")

##### Create Collection (if not exists)

In [14]:
COLLECTION_NAME = "docs"
VECTOR_SIZE = 768

qdrant = QdrantClient(url="http://localhost:6333")

collections = qdrant.get_collections().collections
existing = [c.name for c in collections]

if COLLECTION_NAME not in existing:
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE
        )
    )
    print("Collection created:", COLLECTION_NAME)
else:
    print("Collection already exists:", COLLECTION_NAME)


Collection already exists: docs


##### Embed and Upload to Qdrant

In [18]:
def embed_texts(texts):
    embeddings = model.encode(texts, convert_to_numpy=True)
    return embeddings.tolist()

def normalize(vec):
    vec = np.array(vec, dtype=float)
    norm = np.linalg.norm(vec)
    if norm == 0:
        return vec
    return (vec / norm).tolist()


with open("../text_to_embed.json", "r", encoding="utf-8") as f:
    data = json.load(f)

texts = [d["text"] for d in data]

# Embed in batches
BATCH = 64
points = []

print("Embedding & uploading to Qdrant...")

for i in range(0, len(texts), BATCH):
    batch = data[i:i+BATCH]
    batch_text = [b["text"] for b in batch]

    vectors = embed_texts(batch_text)

    for idx, (item, vector) in enumerate(zip(batch, vectors)):
        vector = normalize(vector)
        p = PointStruct(
            id=i + idx,
            vector=vector,
            payload={
                "text": item["text"],
                "page_title": item.get("page_title", ""),
                "section_title": item.get("section_title", ""),
                "url": item.get("url", ""),
            }
        )
        points.append(p)

    # Upload batch
    qdrant.upsert(
        collection_name=COLLECTION_NAME,
        points=points
    )
    points = []

print("Done uploading.")

Embedding & uploading to Qdrant...
Done uploading.


In [22]:
info = qdrant.get_collection(collection_name="docs")
count = info.points_count

print("Total points in collection:", count)

Total points in collection: 86


##### Query + Return Top Chunks

In [19]:
def embed_query(query: str):
    vec = model.encode([query], convert_to_numpy=True)[0]

    vec = np.array(vec, dtype=float)
    norm = np.linalg.norm(vec)
    if norm != 0:
        vec = vec / norm

    return vec.tolist()

In [20]:
def search(query, top_k=10):
    qvec = normalize(embed_query(query))

    res = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=qvec,
        limit=top_k,
    )

    points = getattr(res, "points", res)

    results = []
    for r in points:
        results.append({
            "id": r.id,
            "score": r.score,
            "text": r.payload.get("text", ""),
            "page_title": r.payload.get("page_title", ""),
            "section_title": r.payload.get("section_title", ""),
            "url": r.payload.get("url", "")
        })

    return results


In [21]:
# Test
query = "users"
matches = search(query, top_k=10)

for i, m in enumerate(matches, 1):
    print(f"\n### Result {i}")
    print("Score:", m["score"])
    print(m["text"])


### Result 1
Score: 0.40326875
Security File: This section ensures user rights are safeguarded. It covers privacy policies, data protection measures, and guidelines for responsible usage, promoting a secure environment for all.

### Result 2
Score: 0.38889584
Ensuring Safety: The application ensures security through encryption, secure login credentials, and regular updates. We prioritize user privacy and conduct security audits to maintain a safe learning environment for all users.

### Result 3
Score: 0.36813083
Give teachers the access to easily manage the room: Hosts decide who can enter, mute or remove participants, limit screen sharing, and lock rooms, ensuring every virtual class stays organized and disruption-free.

### Result 4
Score: 0.36078298
Posts: Tool that simplifies the exchange of posts between teachers and parents, creating a streamlined platform that enhances the distribution of pertinent information. This fosters efficient communication within the school community, 